# 01 — Data Collection & Audit
**Project:** Climate-Driven Solar Energy Analytics  
**Purpose:** Verify the environment, confirm all NASA POWER city files are present and valid, and understand the raw file structure before any analysis begins.

---
### Why this notebook exists
This is a **data audit** notebook — no transformations, no plots, no models. The only goal is to answer:
1. Is our Python environment correctly set up?
2. Are all 15 city CSV files present and non-empty?
3. Do we understand the NASA POWER file format (header rows, column names, record count)?

Skipping this step is how data scientists waste days debugging problems that could have been caught in five minutes.

In [10]:
# ── 1. Environment Check ───────────────────────────────────────────────────
# Import every library used across the ENTIRE project here.
# A single ImportError now is far better than a crash in notebook 09.

import sys
import os
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

print(f"Python   : {sys.version.split()[0]}")
print(f"NumPy    : {np.__version__}")
print(f"Pandas   : {pd.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")
print(f"Seaborn  : {sns.__version__}")
print(f"\nExecutable: {sys.executable}")
print("\n✅ Core environment OK")

Python   : 3.11.9
NumPy    : 2.4.6
Pandas   : 3.0.3
Matplotlib: 3.10.9
Seaborn  : 0.13.2

Executable: /Users/koushikram/Documents/Climate-Driven-Solar-Energy-Analytics/venv/bin/python

✅ Core environment OK


In [11]:
# ── 2. ML Libraries (needed from notebook 08 onward) ──────────────────────
# Check now so we know what to pip-install before we reach those notebooks.

try:
    import sklearn; print(f"scikit-learn : {sklearn.__version__}  ✅")
except ImportError:
    print("scikit-learn : NOT INSTALLED  ❌  →  pip install scikit-learn")

try:
    import xgboost; print(f"xgboost      : {xgboost.__version__}  ✅")
except ImportError:
    print("xgboost      : NOT INSTALLED  ❌  →  pip install xgboost")

try:
    import shap; print(f"shap         : {shap.__version__}  ✅")
except ImportError:
    print("shap         : NOT INSTALLED  ❌  →  pip install shap")

try:
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    import statsmodels; print(f"statsmodels  : {statsmodels.__version__}  ✅")
except ImportError:
    print("statsmodels  : NOT INSTALLED  ❌  →  pip install statsmodels")

    import sys
print(sys.executable)

scikit-learn : 1.9.0  ✅
xgboost      : 3.2.0  ✅
shap         : 0.51.0  ✅
statsmodels  : 0.14.6  ✅
/Users/koushikram/Documents/Climate-Driven-Solar-Energy-Analytics/venv/bin/python


In [12]:
# ── 3. Project-wide Constants ──────────────────────────────────────────────
# Define these ONCE here. All other notebooks import or replicate this block.
# Keeping constants in one place prevents accidental inconsistencies.

RAW_DIR = "../data/raw"
TARGET  = "ALLSKY_SFC_SW_DWN"   # kWh/m²/day — our prediction target

CITY_NAMES = [
    "Ahmedabad", "Bengaluru", "Bhopal", "Bhubaneswar", "Chandigarh",
    "Chennai", "Delhi", "Guwahati", "Hyderabad", "Jaipur",
    "Kochi", "Kolkata", "Mangalore", "Mumbai", "Pune"
]

FEATURE_COLS = ["T2M", "T2M_MAX", "T2M_MIN", "RH2M",
                "PS", "WS10M", "CLOUD_AMT", "PRECTOTCORR"]

# Indian Meteorological Department season classification
SEASON_MAP = {
    1: "Winter",       2: "Winter",       12: "Winter",
    3: "Pre-Monsoon",  4: "Pre-Monsoon",   5: "Pre-Monsoon",
    6: "Monsoon",      7: "Monsoon",       8: "Monsoon",   9: "Monsoon",
    10: "Post-Monsoon", 11: "Post-Monsoon"
}

NASA_SKIPROWS = 17   # NASA POWER CSVs have a 17-line metadata header

print("Constants defined:")
print(f"  Cities     : {len(CITY_NAMES)}")
print(f"  Features   : {FEATURE_COLS}")
print(f"  Target     : {TARGET}")
print(f"  NASA header: first {NASA_SKIPROWS} rows skipped")

Constants defined:
  Cities     : 15
  Features   : ['T2M', 'T2M_MAX', 'T2M_MIN', 'RH2M', 'PS', 'WS10M', 'CLOUD_AMT', 'PRECTOTCORR']
  Target     : ALLSKY_SFC_SW_DWN
  NASA header: first 17 rows skipped


In [13]:
# ── 4. Inspect the NASA POWER File Header ─────────────────────────────────
# Understanding WHY skiprows=17 is correct — not just blindly applying it.
# Print the first 20 lines of one file to see the structure.

sample_path = os.path.join(RAW_DIR, "Bengaluru.csv")

with open(sample_path) as f:
    lines = f.readlines()

print(f"Total lines in Bengaluru.csv: {len(lines)}")
print("=" * 70)
for i, line in enumerate(lines[:20]):
    marker = "   ◄── actual data column headers" if i == 17 else ""
    print(f"Line {i:02d}: {line.rstrip()}{marker}")

Total lines in Bengaluru.csv: 2210
Line 00: -BEGIN HEADER-
Line 01: NASA/POWER Source Native Resolution Daily Data
Line 02: Dates (month/day/year): 01/01/2019 through 12/31/2024 in LST
Line 03: Location: latitude  12.9716   longitude 77.5946
Line 04: elevation from MERRA-2: Average for 0.5 x 0.625 degree lat/lon region = 841.72 meters
Line 05: The value for missing source data that cannot be computed or is outside of the sources availability range: -999
Line 06: parameter(s):
Line 07: T2M                   MERRA-2 Temperature at 2 Meters (C)
Line 08: T2M_MAX               MERRA-2 Temperature at 2 Meters Maximum (C)
Line 09: T2M_MIN               MERRA-2 Temperature at 2 Meters Minimum (C)
Line 10: RH2M                  MERRA-2 Relative Humidity at 2 Meters (%)
Line 11: PS                    MERRA-2 Surface Pressure (kPa)
Line 12: WS10M                 MERRA-2 Wind Speed at 10 Meters (m/s)
Line 13: CLOUD_AMT             CERES SYN1deg Cloud Amount (%)
Line 14: PRECTOTCORR           MERRA

In [14]:
# ── 5. File Audit — All 15 Cities ─────────────────────────────────────────
# For every city: confirm file exists, loads cleanly, has correct dimensions.
# Expected: 2192 rows (2019–2024 inclusive) × 12 columns.
#
# KEY FIX vs original code:
# The original used os.listdir() which picked up download_log.csv.
# Calling skiprows=17 on that file (which has no NASA header) raised:
#   EmptyDataError: No columns to parse from file
# Solution: explicitly list only the known 15 city names.

EXPECTED_ROWS = 2192
EXPECTED_COLS = 12

audit_records = []

for city in CITY_NAMES:
    path = os.path.join(RAW_DIR, f"{city}.csv")
    if not os.path.exists(path):
        audit_records.append({"City": city, "Rows": None, "Cols": None,
                               "Status": "❌ FILE NOT FOUND"})
        continue
    try:
        df_check = pd.read_csv(path, skiprows=NASA_SKIPROWS)
        rows, cols = df_check.shape
        row_ok = rows == EXPECTED_ROWS
        col_ok = cols == EXPECTED_COLS
        status = "✅ OK" if (row_ok and col_ok) else f"⚠️  Expected {EXPECTED_ROWS}r×{EXPECTED_COLS}c"
        audit_records.append({"City": city, "Rows": rows, "Cols": cols, "Status": status})
    except Exception as e:
        audit_records.append({"City": city, "Rows": None, "Cols": None, "Status": f"❌ {e}"})

audit_df = pd.DataFrame(audit_records)
print(audit_df.to_string(index=False))
print(f"\nFiles OK: {(audit_df['Status'] == '✅ OK').sum()} / {len(CITY_NAMES)}")

       City  Rows  Cols Status
  Ahmedabad  2192    12   ✅ OK
  Bengaluru  2192    12   ✅ OK
     Bhopal  2192    12   ✅ OK
Bhubaneswar  2192    12   ✅ OK
 Chandigarh  2192    12   ✅ OK
    Chennai  2192    12   ✅ OK
      Delhi  2192    12   ✅ OK
   Guwahati  2192    12   ✅ OK
  Hyderabad  2192    12   ✅ OK
     Jaipur  2192    12   ✅ OK
      Kochi  2192    12   ✅ OK
    Kolkata  2192    12   ✅ OK
  Mangalore  2192    12   ✅ OK
     Mumbai  2192    12   ✅ OK
       Pune  2192    12   ✅ OK

Files OK: 15 / 15


In [15]:
# ── 6. Download Log Summary ────────────────────────────────────────────────
# The download_log.csv has a DIFFERENT structure — no NASA 17-line header.
# This is the file that caused the EmptyDataError in the original notebook 04.

log = pd.read_csv(os.path.join(RAW_DIR, "download_log.csv"))
print(f"Download log entries : {len(log)}")
print(f"Statuses             : {log['status'].value_counts().to_dict()}")
print(f"Validations          : {log['validation_reason'].value_counts().to_dict()}")
print(f"Total size collected : {log['size_bytes'].sum() / 1e6:.1f} MB")
print()
log[["city", "status", "rows", "size_bytes", "validation_reason"]].head(16)

Download log entries : 15
Statuses             : {'skipped': 15}
Validations          : {'ok': 15}
Total size collected : 2.1 MB



,city,status,rows,size_bytes,validation_reason
0,Bengaluru,skipped,2192,138545,ok
1,Chennai,skipped,2192,140528,ok
2,Hyderabad,skipped,2192,138144,ok
3,Kochi,skipped,2192,141598,ok
4,Mangalore,skipped,2192,139349,ok
5,Mumbai,skipped,2192,139693,ok
6,Pune,skipped,2192,138193,ok
7,Ahmedabad,skipped,2192,138971,ok
8,Delhi,skipped,2192,137462,ok
9,Chandigarh,skipped,2192,137771,ok


In [16]:
# ── 7. Dataset Metadata ────────────────────────────────────────────────────

with open(os.path.join(RAW_DIR, "dataset_metadata.json")) as f:
    meta = json.load(f)
print(json.dumps(meta, indent=2))

{
  "dataset_name": "Indian Urban Climate Data (NASA POWER)",
  "description": "Daily meteorological observations for 15 major Indian cities sourced from NASA POWER Prediction Of Worldwide Energy Resources API.",
  "download_timestamp_utc": "2026-06-05T20:20:31.101713+00:00",
  "date_range": {
    "start": "20190101",
    "end": "20241231"
  },
  "parameters": {
    "T2M": "Temperature at 2m height (\u00b0C)",
    "T2M_MAX": "Daily maximum temperature at 2m (\u00b0C)",
    "T2M_MIN": "Daily minimum temperature at 2m (\u00b0C)",
    "RH2M": "Relative humidity at 2m (%)",
    "PS": "Surface pressure (kPa)",
    "WS10M": "Wind speed at 10m height (m/s)",
    "CLOUD_AMT": "Cloud amount (%)",
    "PRECTOTCORR": "Bias-corrected total precipitation (mm/day)",
    "ALLSKY_SFC_SW_DWN": "All-sky surface shortwave downward irradiance / GHI (kWh/m\u00b2/day)",
    "ALLSKY_SFC_SW_DNI": "All-sky surface shortwave direct normal irradiance / DNI (kWh/m\u00b2/day)"
  },
  "api_source": "https://power.l

In [17]:
try:
    import statsmodels
    print("statsmodels imported successfully")
    print(statsmodels.__version__)
except Exception as e:
    print("ERROR:")
    print(e)

statsmodels imported successfully
0.14.6


In [18]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

print("VIF import successful")

VIF import successful


---
## ✅ Collection Summary

| Item | Value |
|---|---|
| Source | NASA POWER API |
| Cities | 15 Indian cities |
| Time range | 2019–2024 (6 years) |
| Records per city | 2,192 daily rows |
| Total records | 32,880 rows |
| Features | 12 columns (YEAR, MO, DY + 9 climate variables) |
| Target variable | `ALLSKY_SFC_SW_DWN` — All Sky Surface Shortwave Downward Irradiance (kWh/m²/day) |
| Missing values | None |
| Header rows to skip | 17 (NASA POWER metadata format) |

**Critical note for all downstream notebooks:**  
Always load city files using `CITY_NAMES` list, never `os.listdir()`.  
`os.listdir()` includes `download_log.csv` which has no NASA header → `EmptyDataError`.

**Next → `02_data_understanding.ipynb`**